# Analyse Att — v2 (minimal)

Une seule chose, rien d'autre :

1. Lire chaque tag Att minute par minute depuis OIA.
2. Ajouter une colonne `pas` = repère // 100 (partie entière de la division par 100).
3. Réduire : ne garder que les pas, compter le nombre de minutes entre chaque changement de pas.

Pas de table de référence process, pas de notion de défaut, pas de reconstruction d'opération. Une table par tag : `pas`, `debut`, `duree_min`.

In [1]:
import json
import sys

import numpy as np
import pandas as pd

from tools.OI_class_OP import OI_DataProcessor

pd.set_option('display.max_columns', None)

for _mod in list(sys.modules):
    if _mod == 'att_tags_config' or _mod.startswith('att_tags_config.'):
        del sys.modules[_mod]

from att_tags_config import TAGS as tags, is_batch

batch_tags = [t for t in tags if is_batch(t)]

processor = OI_DataProcessor(
    url_base='https://oianalytics-100.optimistik.fr/api/oianalytics/time-values/query?',
    start='2020-01-01',
    end='2026-12-31',
    tags_selected=tags,
    tags_other=[],
    interval='PT01M',
    verbose=False,
    agg='MEAN',
)
processor.merge()
print(f"{len(batch_tags)} tags batch sur {len(tags)}")


21 tags batch sur 28


In [2]:
# Avec MEAN, un bucket qui chevauche une transition peut fabriquer une
# valeur intermediaire (ex: 1638) qui ne correspond a aucun etat reel.
# Correctif : une valeur "tenue" (plateau reel) apparait forcement
# plusieurs fois dans l'historique, alors qu'une valeur fabriquee par MEAN
# sur une transition est un melange unique, presque jamais reproduit a
# l'identique -- on construit donc, par tag, l'ensemble des valeurs vues
# au moins 2 fois, et on corrige chaque valeur brute vers la plus proche
# de cet ensemble avant de calculer le pas. Sauvegarde dans un JSON pour
# ne pas avoir a la recalculer a chaque fois.
CHEMIN_VALEURS_CONNUES = 'documents/confidentiel/att_valeurs_connues.json'
MIN_OCCURRENCES = 2


def construire_valeurs_connues(min_occurrences: int = MIN_OCCURRENCES) -> dict:
    valeurs_connues = {}
    for tag_def in batch_tags:
        label, nom = tag_def['tag'], tag_def['nom']
        if nom not in processor.data.columns:
            continue
        comptages = processor.data[nom].value_counts()
        valeurs_connues[label] = sorted(comptages[comptages >= min_occurrences].index.tolist())
    return valeurs_connues


valeurs_connues = construire_valeurs_connues()
with open(CHEMIN_VALEURS_CONNUES, 'w', encoding='utf-8') as f:
    json.dump(valeurs_connues, f, indent=2)

print(f"{sum(len(v) for v in valeurs_connues.values())} valeurs connues sauvegardees dans {CHEMIN_VALEURS_CONNUES}")


115089 valeurs connues sauvegardees dans documents/confidentiel/att_valeurs_connues.json


In [3]:
def corriger_serie(serie: pd.Series, valeurs: list) -> pd.Series:
    """
    Remplace chaque valeur de `serie` par la plus proche dans `valeurs`
    (liste triee) -- corrige les valeurs fabriquees par MEAN sur une
    transition, qui ne correspondent a aucun etat reel.
    """
    valeurs_arr = np.asarray(valeurs)
    brutes = serie.to_numpy()
    idx = np.searchsorted(valeurs_arr, brutes)
    idx_avant = np.clip(idx - 1, 0, len(valeurs_arr) - 1)
    idx_apres = np.clip(idx, 0, len(valeurs_arr) - 1)
    avant = valeurs_arr[idx_avant]
    apres = valeurs_arr[idx_apres]
    plus_proche = np.where(np.abs(brutes - avant) <= np.abs(apres - brutes), avant, apres)
    return pd.Series(plus_proche, index=serie.index)


def reduire_en_pas(nom_colonne: str, valeurs: list) -> pd.DataFrame:
    """
    Lit la colonne brute (1 point/minute), corrige chaque valeur vers la
    plus proche valeur connue (cf. corriger_serie), ajoute pas = repere //
    100, puis reduit en plateaux : une ligne par pas, avec sa duree en
    minutes (debut du pas suivant - debut de ce pas). Rien d'autre.

    Repere < 100 exclu : aucun pas reel ne descend en dessous de 100 (pas1
    commence a 100) -- une valeur brute sous ce seuil (le plus souvent 0)
    est un artefact transitoire de changement de pas, pas un etat reel.
    """
    serie = processor.data[[nom_colonne]].rename(columns={nom_colonne: 'repere'}).copy()
    serie = serie[serie['repere'].notna() & (serie['repere'] >= 100)]
    serie['repere'] = corriger_serie(serie['repere'], valeurs)
    serie['pas'] = (serie['repere'] // 100).astype(int)

    nouveau_pas = serie['pas'] != serie['pas'].shift()
    serie['pas_id'] = nouveau_pas.cumsum()

    serie_reset = serie.reset_index()
    colonne_temps = serie_reset.columns[0]  # nom reel de l'index (variable selon la source)
    serie_reset = serie_reset.rename(columns={colonne_temps: 'debut'})

    plateaux = (
        serie_reset
        .groupby('pas_id')
        .agg(pas=('pas', 'first'), debut=('debut', 'first'))
        .reset_index(drop=True)
    )
    plateaux['fin'] = plateaux['debut'].shift(-1)
    plateaux.loc[plateaux.index[-1], 'fin'] = serie.index[-1]
    plateaux['duree_min'] = (plateaux['fin'] - plateaux['debut']).dt.total_seconds() / 60

    return plateaux[['pas', 'debut', 'duree_min']]


tables_pas = {}
for tag_def in batch_tags:
    label, nom = tag_def['tag'], tag_def['nom']
    if nom not in processor.data.columns:
        continue
    tables_pas[label] = reduire_en_pas(nom, valeurs_connues[label])

print(f"{len(tables_pas)} tables construites")


21 tables construites


In [4]:
# Changer cette valeur pour inspecter un autre tag.
exemple = 'PU3230VA_Att'
tables_pas[exemple].head(50)

,pas,debut,duree_min
0,4,2020-01-01 00:00:00+00:00,142.0
1,5,2020-01-01 02:22:00+00:00,5.0
2,6,2020-01-01 02:27:00+00:00,10.0
3,7,2020-01-01 02:37:00+00:00,59.0
4,8,2020-01-01 03:36:00+00:00,10.0
5,9,2020-01-01 03:46:00+00:00,54.0
6,7,2020-01-01 04:40:00+00:00,1.0
7,4,2020-01-01 04:41:00+00:00,253.0
8,5,2020-01-01 08:54:00+00:00,4.0
9,6,2020-01-01 08:58:00+00:00,10.0


In [5]:
# D'ou viennent les pas=0 ? Regarder les valeurs brutes de repere qui y menent.
serie_diag = processor.data[[next(t for t in batch_tags if t['tag'] == exemple)['nom']]].copy()
serie_diag.columns = ['repere']
serie_diag = serie_diag[serie_diag['repere'].notna()]

sous_100 = serie_diag[serie_diag['repere'] < 100]
print(f"{len(sous_100):,} lignes avec repere < 100 sur {len(serie_diag):,} lignes valides")
sous_100['repere'].value_counts()


32,389 lignes avec repere < 100 sur 3,431,973 lignes valides


repere
0.000000     32378
95.000000        2
31.428571        1
27.000000        1
8.633333         1
51.333333        1
14.666667        1
65.333333        1
31.166667        1
56.833333        1
23.833333        1
Name: count, dtype: int64

## Compter les opérations

Règle : si on est arrivé à un pas proche de la fin de la séquence, puis
qu'on revient à un pas proche du début, on change d'opération. La plage
réelle de pas (min/max documenté) vient de `pas_reference` (juste
`pas_num`, rien d'autre — pas de plage de repère, pas de code_court, pas
de défaut).

In [6]:
TOLERANCE_PAS_ATTENTE = 1  # marge absolue sur le pas d'attente (variabilite observee, ex: OP2410 -> pas4 ou pas5)
FRACTION_FIN = 0.2  # 20% du haut de la plage reelle = "proche de la fin"


def pas_reel_min_max(pas_reference: pd.DataFrame) -> tuple:
    """
    pas_num min/max des SEULS pas reels de la sequence -- exclut MODES (pas
    transverse, commun a toute l'operation, jamais un vrai pas de
    sequence) et les pas DEF.* (defaut) : sans cette exclusion, pas_max
    est gonfle artificiellement (ex: OP1540 a 9 pas au total, dont 8=DEF et
    9=MODES -- le vrai dernier pas de la sequence est le 7).
    """
    code_court = pas_reference['code_court'].astype(str).str.upper()
    est_transverse_ou_defaut = (code_court == 'MODES') | code_court.str.startswith('DEF')
    pas_reels = pas_reference.loc[~est_transverse_ou_defaut, 'pas_num']
    return int(pas_reels.min()), int(pas_reels.max())


def ajouter_operation(
    table: pd.DataFrame,
    pas_min: int,
    pas_max: int,
    pas_attente: int,
    tolerance: int = TOLERANCE_PAS_ATTENTE,
    fraction_fin: float = FRACTION_FIN,
) -> pd.DataFrame:
    """
    Ajoute une colonne 'operation' : compteur qui s'incremente des qu'une
    regression revient a un pas <= pas_attente + tolerance, MAIS
    seulement si l'operation en cours a deja atteint un pas proche de la
    fin reelle (pas_max) auparavant.

    Les DEUX conditions sont necessaires (une seule ne suffit pas) :
    - "retour au pas d'attente" seul declenche a tort sur les petites
      oscillations juste apres un redemarrage, avant toute progression
      (ex: un chargement qui bafouille pas4<->pas5 en tout debut de
      cycle) -- d'ou des "operations" de 1 minute observees sans le
      garde-fou "a atteint la fin" ;
    - "a atteint la fin" seul (sans regarder OU on retombe) declenche a
      tort sur de simples oscillations tardives (ex: pas17->pas14) qui ne
      reviennent pas au vrai point d'entree.
    """
    etendue = pas_max - pas_min
    proche_fin = table['pas'].to_numpy() >= pas_max - fraction_fin * etendue
    seuil_attente = pas_attente + tolerance
    pas = table['pas'].to_numpy()

    operation = np.empty(len(table), dtype=int)
    num_operation = 1
    a_atteint_la_fin = False
    for i in range(len(table)):
        if i > 0 and pas[i] < pas[i - 1] and pas[i] <= seuil_attente and a_atteint_la_fin:
            num_operation += 1
            a_atteint_la_fin = False
        if proche_fin[i]:
            a_atteint_la_fin = True
        operation[i] = num_operation

    table = table.copy()
    table['operation'] = operation
    return table


SEUIL_FUSION = 0.75  # une "operation" de duree < 75% du temps de reference est fusionnee avec la precedente


def fusionner_courtes(table: pd.DataFrame, temps_reference, seuil: float = SEUIL_FUSION) -> pd.DataFrame:
    """
    Filet de securite complementaire : fusionne avec l'operation
    precedente toute operation dont la duree TOTALE (attente comprise)
    est < seuil * temps de reference. Sans temps de reference connu, ne
    fait rien (colonne 'fusionnee' = False partout).

    ATTENTION : utilise bien la duree TOTALE, PAS la duree hors pas
    d'attente -- constate sur OP3230, dont le pas d'attente (pas4,
    ATT.TRANSF.) est lui-meme une vraie attente longue (286-633 min), pas
    juste une porte de lancement breve. En l'excluant ici, chaque morceau
    semblait artificiellement court et se faisait fusionner a tort avec
    le precedent, alors que sa duree totale etait largement suffisante.
    Seul l'affichage (resume_ope) continue d'exclure le pas d'attente des
    statistiques, comme demande pour OP2340.

    Ajoute une colonne 'fusionnee' (bool) : True pour toutes les lignes
    dont l'operation finale resulte de la fusion de 2 morceaux ou plus --
    sert a mesurer le % d'operations "propres" (jamais fusionnees).
    """
    if not temps_reference:
        table = table.copy()
        table['fusionnee'] = False
        return table

    table = table.copy()
    duree_par_ope = table.groupby('operation')['duree_min'].sum()
    toutes_les_ope = sorted(table['operation'].unique())
    duree_par_ope = duree_par_ope.reindex(toutes_les_ope, fill_value=0.0)

    fusion = {}
    derniere_conservee = toutes_les_ope[0]
    fusion[derniere_conservee] = derniere_conservee
    for op in toutes_les_ope[1:]:
        if duree_par_ope[op] < seuil * temps_reference:
            fusion[op] = fusion[derniere_conservee]
        else:
            fusion[op] = op
            derniere_conservee = op

    table['operation'] = table['operation'].map(fusion)
    compte_morceaux = pd.Series(fusion).value_counts()  # nb de morceaux bruts fusionnes par valeur finale
    table['fusionnee'] = table['operation'].map(lambda v: compte_morceaux.get(v, 1) > 1)
    table['operation'] = table['operation'].rank(method='dense').astype(int)  # renumerote 1,2,3...
    return table


plage_pas = {}  # {tag: (pas_min, pas_max)}, garde pour reutilisation plus bas

for label in tables_pas:
    tag_def = next(t for t in batch_tags if t['tag'] == label)
    pas_reference = pd.read_csv(tag_def['pas_reference'])
    pas_min, pas_max = pas_reel_min_max(pas_reference)
    plage_pas[label] = (pas_min, pas_max)

    pas_attente = tag_def.get('pas_attente', pas_min)
    tables_pas[label] = ajouter_operation(tables_pas[label], pas_min, pas_max, pas_attente)
    tables_pas[label] = fusionner_courtes(tables_pas[label], tag_def.get('temps_reference_min'))

print(f"{tables_pas[exemple]['operation'].max()} operations detectees pour {exemple}")
tables_pas[exemple].head(50)


6004 operations detectees pour PU3230VA_Att


,pas,debut,duree_min,operation,fusionnee
0,4,2020-01-01 00:00:00+00:00,142.0,1,False
1,5,2020-01-01 02:22:00+00:00,5.0,1,False
2,6,2020-01-01 02:27:00+00:00,10.0,1,False
3,7,2020-01-01 02:37:00+00:00,59.0,1,False
4,8,2020-01-01 03:36:00+00:00,10.0,1,False
5,9,2020-01-01 03:46:00+00:00,54.0,1,False
6,7,2020-01-01 04:40:00+00:00,1.0,1,False
7,4,2020-01-01 04:41:00+00:00,253.0,2,False
8,5,2020-01-01 08:54:00+00:00,4.0,2,False
9,6,2020-01-01 08:58:00+00:00,10.0,2,False


In [7]:
# Verification : pas_max reel (hors MODES/DEF) par rapport au pas_num brut
# maximum du CSV, pour reperer une eventuelle anomalie a l'oeil.
verif_plage = []
for tag_def in batch_tags:
    label = tag_def['tag']
    pas_reference = pd.read_csv(tag_def['pas_reference'])
    pas_min, pas_max = pas_reel_min_max(pas_reference)
    pas_num_max_brut = int(pas_reference['pas_num'].max())
    code_court_brut = pas_reference.loc[pas_reference['pas_num'] == pas_num_max_brut, 'code_court'].iloc[0]

    verif_plage.append({
        'Tag': label,
        'Opération': tag_def.get('operation', ''),
        'pas_min (réel)': pas_min,
        'pas_max (réel)': pas_max,
        'pas_num max brut (CSV)': pas_num_max_brut,
        'code_court du pas max brut': code_court_brut,
    })

pd.DataFrame(verif_plage)


,Tag,Opération,pas_min (réel),pas_max (réel),pas_num max brut (CSV),code_court du pas max brut
0,PU1410VA_Att,OP1410,1,23,24,MODES
1,PU1420VA_Att,OP1420,1,14,16,MODES
2,PU1430VA_Att,OP1430,1,27,29,MODES
3,PU1510VA_Att,OP1510,1,10,10,HOLD
4,PU1520VA_Att,OP1520,1,13,13,HOLD
5,PU1530VA_Att,OP1530,1,9,9,HOLD
6,PU1540VA_Att,OP1540,1,17,17,HOLD
7,PU2810VA_Att,OP2810,1,14,16,MODES
8,PU2310VA_Att,OP2310,1,16,18,MODES
9,PU2320VA_Att,OP2320,1,21,23,MODES


In [8]:
def exclure_bornes(table: pd.DataFrame) -> pd.DataFrame:
    """
    Retire la 1ere et la derniere operation (chronologiquement) : la
    periode interrogee commence/finit forcement en cours de cycle, ces
    deux operations sont donc potentiellement tronquees. Rien d'autre.
    """
    if table['operation'].nunique() < 3:
        return table
    premiere = table['operation'].min()
    derniere = table['operation'].max()
    return table[~table['operation'].isin([premiere, derniere])]


In [9]:
# Temps cumule depuis le debut de l'operation en cours (repart a 0 a
# chaque changement d'operation). Rien d'autre.
for label in tables_pas:
    tables_pas[label]['temps_cumule_min'] = tables_pas[label].groupby('operation')['duree_min'].cumsum()

tables_pas[exemple].head(50)


,pas,debut,duree_min,operation,fusionnee,temps_cumule_min
0,4,2020-01-01 00:00:00+00:00,142.0,1,False,142.0
1,5,2020-01-01 02:22:00+00:00,5.0,1,False,147.0
2,6,2020-01-01 02:27:00+00:00,10.0,1,False,157.0
3,7,2020-01-01 02:37:00+00:00,59.0,1,False,216.0
4,8,2020-01-01 03:36:00+00:00,10.0,1,False,226.0
5,9,2020-01-01 03:46:00+00:00,54.0,1,False,280.0
6,7,2020-01-01 04:40:00+00:00,1.0,1,False,281.0
7,4,2020-01-01 04:41:00+00:00,253.0,2,False,253.0
8,5,2020-01-01 08:54:00+00:00,4.0,2,False,257.0
9,6,2020-01-01 08:58:00+00:00,10.0,2,False,267.0


In [10]:
# Matrice : lignes = operations, colonnes = pas (1 a 32), cases = temps
# total (min) passe dans ce pas pour cette operation.
table = tables_pas[exemple]
matrice = table.pivot_table(index='operation', columns='pas', values='duree_min', aggfunc='sum', fill_value=0)
matrice = matrice.reindex(columns=range(1, 33), fill_value=0)
matrice


pas,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
operation,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,142.0,5.0,10.0,60.0,10.0,54.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0.0,0.0,0.0,253.0,4.0,10.0,61.0,8.0,55.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,0.0,0.0,0.0,250.0,5.0,9.0,60.0,10.0,55.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,0.0,0.0,1.0,248.0,27.0,10.0,61.0,11.0,53.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
5,0.0,0.0,0.0,488.0,5.0,9.0,60.0,10.0,54.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6000,0.0,0.0,0.0,240.0,6.0,4.0,60.0,12.0,54.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6001,0.0,0.0,1.0,683.0,6.0,5.0,604.0,13.0,1669.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
6002,0.0,0.0,1.0,294.0,7.0,3.0,60.0,12.0,209.0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [11]:
# Temps d'une operation = somme de duree_min SANS le pas d'attente/init
# (toujours le pas_num le plus bas documente dans pas_reference -- l'
# "ATTENTE de lancement", qui n'est pas du temps de production reel).
# Rien d'autre.
pas_min_exemple, _ = plage_pas[exemple]

duree_operation = (
    tables_pas[exemple][tables_pas[exemple]['pas'] != pas_min_exemple]
    .groupby('operation')['duree_min']
    .sum()
    .rename('duree_operation_min')
)
duree_operation.head(20)


operation
1     281.0
2     391.0
3     389.0
4     411.0
5     626.0
6     392.0
7     389.0
8     389.0
9     392.0
10    414.0
11    384.0
12    393.0
13    410.0
14    444.0
15    524.0
16    842.0
17    391.0
18    420.0
19    376.0
20    391.0
Name: duree_operation_min, dtype: float64

In [12]:
# Matrice globale : lignes = les operations (tags), colonnes = pas (1 a
# 32), cases = temps MINIMUM observe pour ce pas sur tout l'historique de
# cette operation (tous cycles confondus, pas par instance, 1ere/derniere
# operation exclues). 0 quand ce pas n'existe pas pour cette operation
# (pas NaN).
minimums_par_tag = {}
for label in tables_pas:
    minimums_par_tag[label] = exclure_bornes(tables_pas[label]).groupby('pas')['duree_min'].min()

matrice_min_globale = pd.DataFrame(minimums_par_tag).T.reindex(columns=range(1, 33)).fillna(0)
matrice_min_globale


pas,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32
PU1410VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU1420VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU1430VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0
PU1510VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU1520VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU1530VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU1540VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
PU2810VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,12.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU2310VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
PU2320VA_Att,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [13]:
# Un tableau, une ligne par operation (tag) : N ope, temps moyen/median
# d'une operation (somme de tous les pas SAUF le pas d'attente), les 5
# temps les plus petits, le temps moyen d'attente, le temps de reference
# connu de l'exploitant, le % d'operations "propres" (jamais fusionnees
# par fusionner_courtes) et l'ecart de la moyenne des 5 plus petits vs
# la reference. 1ere/derniere operation exclues (tronquees).
resume_rows = []
for label in tables_pas:
    table = exclure_bornes(tables_pas[label])
    pas_min_tag, _ = plage_pas[label]
    tag_def = next(t for t in batch_tags if t['tag'] == label)
    pas_attente = tag_def.get('pas_attente', pas_min_tag)
    reference = tag_def.get('temps_reference_min')

    duree_par_ope = table[table['pas'] != pas_attente].groupby('operation')['duree_min'].sum()
    attente_par_ope = table[table['pas'] == pas_attente].groupby('operation')['duree_min'].sum()

    # % d'operations propres (jamais fusionnees) : une operation est
    # "propre" si AUCUNE de ses lignes n'est marquee fusionnee=True.
    propre_par_ope = ~table.groupby('operation')['fusionnee'].any()
    pct_propres = round(propre_par_ope.mean() * 100, 1)

    cinq_plus_petits = duree_par_ope.nsmallest(5)
    moyenne_5_petits = cinq_plus_petits.mean()
    ecart_pct = round((moyenne_5_petits - reference) / reference * 100, 1) if reference else None

    resume_rows.append({
        'Tag': label,
        'Pas attente': pas_attente,
        'N ope': duree_par_ope.shape[0],
        '% ope propres': pct_propres,
        'Temps moyen (min)': round(duree_par_ope.mean(), 0),
        'Temps median (min)': round(duree_par_ope.median(), 0),
        '5 plus petits (min)': sorted(cinq_plus_petits.round(0).tolist()),
        'Ecart moy. 5 petits vs ref (%)': ecart_pct,
        'Attente moyenne (min)': round(attente_par_ope.mean(), 1),
        'Temps réf. exploitant (min)': reference,
    })

resume_ope = pd.DataFrame(resume_rows).set_index('Tag')
resume_ope


,Pas attente,N ope,% ope propres,Temps moyen (min),Temps median (min),5 plus petits (min),Ecart moy. 5 petits vs ref (%),Attente moyenne (min),Temps réf. exploitant (min)
Tag,,,,,,,,,
PU1410VA_Att,4,4649,100.0,730.0,513.0,"[413.0, 413.0, 415.0, 416.0, 416.0]",-2.4,10.5,425
PU1420VA_Att,3,4639,99.2,741.0,524.0,"[336.0, 337.0, 337.0, 339.0, 340.0]",52.2,2.4,222
PU1430VA_Att,2,1544,99.9,2224.0,1681.0,"[980.0, 1017.0, 1029.0, 1057.0, 1062.0]",0.2,6.8,1027
PU1510VA_Att,3,5247,94.9,627.0,465.0,"[15.0, 341.0, 360.0, 367.0, 371.0]",-29.4,35.4,412
PU1520VA_Att,3,5275,55.7,621.0,454.0,"[202.0, 236.0, 261.0, 261.0, 264.0]",-36.7,38.8,387
PU1530VA_Att,3,5276,85.2,610.0,454.0,"[41.0, 293.0, 297.0, 316.0, 316.0]",-39.6,56.5,418
PU1540VA_Att,3,5300,69.2,622.0,461.0,"[72.0, 276.0, 318.0, 331.0, 332.0]",-32.0,35.4,391
PU2810VA_Att,4,891,100.0,3791.0,3153.0,"[807.0, 826.0, 862.0, 1058.0, 1059.0]",4.1,70.2,886
PU2310VA_Att,5,4572,99.8,697.0,519.0,"[429.0, 436.0, 440.0, 440.0, 441.0]",-17.5,55.8,530


In [14]:
# Alerte : operation dont la duree (hors pas d'attente) est trop faible
# par rapport au temps de reference exploitant.
SEUIL_ALERTE = 0.8  # signalee si duree < 50% du temps de reference

alertes_rows = []
for label in tables_pas:
    table = exclure_bornes(tables_pas[label])
    pas_min_tag, _ = plage_pas[label]
    tag_def = next(t for t in batch_tags if t['tag'] == label)
    pas_attente = tag_def.get('pas_attente', pas_min_tag)
    reference = tag_def.get('temps_reference_min')
    if not reference:
        continue

    duree_par_ope = table[table['pas'] != pas_attente].groupby('operation')['duree_min'].sum()
    suspectes = duree_par_ope[duree_par_ope < SEUIL_ALERTE * reference]

    for num_ope, duree in suspectes.items():
        alertes_rows.append({
            'Tag': label,
            'operation': num_ope,
            'duree_operation_min': round(duree, 0),
            'temps_reference_min': reference,
        })

alertes = pd.DataFrame(alertes_rows)
print(f"{len(alertes)} operations signalees (duree < {SEUIL_ALERTE*100:.0f}% de la reference)")
alertes


5952 operations signalees (duree < 80% de la reference)


,Tag,operation,duree_operation_min,temps_reference_min
0,PU1510VA_Att,1616,15.0,412
1,PU1520VA_Att,1641,202.0,387
2,PU1520VA_Att,1751,277.0,387
3,PU1520VA_Att,1754,309.0,387
4,PU1520VA_Att,1757,264.0,387
...,...,...,...,...
5947,PU3310VA_Att,4757,261.0,338
5948,PU3310VA_Att,4840,262.0,338
5949,PU3310VA_Att,5269,268.0,338
5950,PU3310VA_Att,5841,268.0,338


In [15]:
def contexte_operation(label: str, num_operation: int, marge: int = 20) -> pd.DataFrame:
    """
    Lignes de l'operation numero `num_operation` du tag `label`, entourees
    de `marge` pas avant et apres -- pour analyser visuellement une
    operation signalee (alertes).
    """
    table = tables_pas[label]
    indices = table.index[table['operation'] == num_operation]
    debut = max(indices.min() - marge, 0)
    fin = min(indices.max() + marge, table.index.max())
    return table.loc[debut:fin]


# Exemple : la 1ere operation signalee pour PU3230VA_Att.
alertes_3230 = alertes[alertes['Tag'] == 'PU3230VA_Att']
alertes_3230


,Tag,operation,duree_operation_min,temps_reference_min
128,PU3230VA_Att,2,138.0,337
129,PU3230VA_Att,3,139.0,337
130,PU3230VA_Att,4,163.0,337
131,PU3230VA_Att,5,138.0,337
132,PU3230VA_Att,6,139.0,337
...,...,...,...,...
5941,PU3230VA_Att,5996,139.0,337
5942,PU3230VA_Att,5997,139.0,337
5943,PU3230VA_Att,5998,136.0,337
5944,PU3230VA_Att,5999,161.0,337


In [16]:
# Changer le numero d'operation ci-dessous pour en inspecter une autre
# (voir la liste dans alertes_2410 / alertes_3210).
contexte_operation('PU3230VA_Att', int(alertes_2410['operation'].iloc[0]))


NameError: name 'alertes_2410' is not defined

In [ ]:
# Idem pour PU3210VA_Att.
alertes_3210 = alertes[alertes['Tag'] == 'PU3210VA_Att']
alertes_3210


In [ ]:
contexte_operation('PU3210VA_Att', int(alertes_3210['operation'].iloc[13]))


In [ ]:
# Valeurs brutes OIA entre deux dates, plusieurs tags cote a cote.
# Chaque tag n'enregistre un point que quand SA PROPRE valeur change
# (compression/deadband, cf. plateau pas1 de 205 min avec seulement 57
# points bruts) -- en combinant plusieurs tags independants, la plupart
# des cases sont donc NaN par construction. ffill() propage la derniere
# valeur connue pour voir la valeur COURANTE de chaque tag a chaque minute.
tags_bruts = ['PU3210VA_Att', 'PU2410VA_Att', 'PU2420VA_Att', 'PU3220VA_Att']
debut_brut = '2026-03-06 09:50:00'
fin_brut = '2026-03-06 12:20:00'

noms_bruts = [next(t for t in batch_tags if t['tag'] == tag)['nom'] for tag in tags_bruts]
processor.data.loc[debut_brut:fin_brut, noms_bruts].head(30)
